In [47]:
import logging
import os
import time

from dotenv import load_dotenv
import psycopg2
from psycopg2 import OperationalError, sql

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler("etl.log", mode="a"), logging.StreamHandler()],
    force=True,
)

logger = logging.getLogger("ETL_Pipeline")

DB_CONFIG = {
    "dbname": os.getenv("DB_NAME", "bank_db"),
    "user": os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", "postgres"),
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
}

print("Configuration and Logger Initialized.")

Configuration and Logger Initialized.


In [48]:
# Task 2 = Harden Connection with Retries and Rollbacks

import functools

def retry_db_operation(max_retries=3, delay=1):
    """
    Decorator that intercepts database operations, catches pycopg2.OperationalError, 

    Performs an explicit transaction rollback, and retries the operation with a short delay.
    """

    def decorator(func):
        @functools.wraps(func)
        def wrapper(self_or_conn, *args, **kwargs):
            attempts = 0
            while attempts < max_retries:
                try:
                    return func(self_or_conn, *args, **kwargs)
                except OperationalError as e:
                    attempts +=1
                    logger.warning(f"OperationalError encountered on attempt {attempts}/{max_retries}: {e}")

                    conn = getattr(self_or_conn, "conn", self_or_conn)
                    if conn and not conn.closed:
                        try:
                            conn.rollback()
                            logger.info("Excecuted db rollback successfully.")
                        except Exception as rb_err:
                            logger.error(f"Rollback attempt failed: {rb_err}")

                    if attempts >= max_retries:
                        logger.error("Max retries reached.")
                        raise e

                    time.sleep(delay)

        return wrapper
    return decorator


class DBConnection:
    def __init__(self, config):
        self.config = config
        self.conn = None

    def connect(self):
        if not self.conn or self.conn.closed:
            self.conn = psycopg2.connect(**self.config)
            self.conn.autocommit = False
        return self.conn

    @retry_db_operation(max_retries=3, delay=1)
    def execute_query(self, query, params=None, fetch=False):
        """ 
        Executes parameterized queries safely with error wrapping and retries
        """ 
        conn = self.connect()
        with conn.cursor() as cur:
            cur.execute(query, params or ())
            result = cur.fetchall() if fetch else None
        conn.commit()
        return result

    def close(self):
        if self.conn and not self.conn.closed:
            self.conn.close()

logger.info("DBConnection class and hardening decorator created")


2026-09-23 21:38:15,639 [INFO] DBConnection class and hardening decorator created


In [49]:
#Task 1 - Idempotent schema creation and data seeding


def create_scema(db: DBConnection):
    """
    Creates customers, transactions, loans and account_summary tables using idempodent DDL.
    """
    ddl_queries = [
        """ 
        CREATE TABLE IF NOT EXISTS customers (
            customer_id SERIAL PRIMARY KEY,
            name VARCHAR(100) NOT NULL,
            email VARCHAR(100) UNIQUE NOT NULL
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS transactions (
            transaction_id SERIAL PRIMARY KEY,
            customer_id INTEGER REFERENCES customers(customer_id) ON DELETE CASCADE,
            amount NUMERIC(12, 2) NOT NULL,
            transaction_date DATE NOT NULL DEFAULT CURRENT_DATE
        )
        """,
        """ 
        CREATE TABLE IF NOT EXISTS loans (
            loan_id SERIAL PRIMARY KEY,
            customer_id INTEGER REFERENCES customers(customer_id) ON DELETE CASCADE,
            principal NUMERIC(12, 2) NOT NULL,
            interest_rate NUMERIC(5, 2) NOT NULL,
            status TEXT NOT NULL CHECK (status IN ('active', 'paid_off', 'defaulted')),
            start_date DATE NOT NULL DEFAULT CURRENT_DATE
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS account_summary (
            customer_id INTEGER PRIMARY KEY REFERENCES customers(customer_id) ON DELETE CASCADE,
            total_transactions NUMERIC(12, 2) DEFAULT 0.00,
            total_loan_exposure NUMERIC(12, 2) DEFAULT 0.00,
            active_loan_count INTEGER DEFAULT 0,
            category VARCHAR(20) NOT NULL,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """
    ]

    for ddl in ddl_queries:
        db.execute_query(ddl)

    logger.info("Idempodent schema created / verified successfully.")


def seed_data(db: DBConnection):
    """ 
    Seeds base sample data into customers, transactions, and loan tables.
    """
    customers = [
        (1, "Alice Smith", "alice@example.com"),
        (2, "Bob Jones", "bob@example.com"),
        (3, "Charlie Brown", "charlie@example.com"),
    ]

    for cid, name, email in customers:
        query = """ 
            INSERT INTO customers (customer_id, name, email) 
            VALUES (%s, %s, %s)
            ON CONFLICT (customer_id) DO UPDATE SET name = EXCLUDED.name, email = EXCLUDED.email;
        """
        db.execute_query(query, (cid, name, email))

    transactions = [
        (1, 1, 15000.00),
        (2, 1, 6000.00),
        (3, 2, 7000.00),
        (4, 3, 1000.00),
    ]
    for tid, cid, amt in transactions:
        query = """
            INSERT INTO transactions (transaction_id, customer_id, amount)
            VALUES (%s, %s, %s)
            ON CONFLICT (transaction_id) DO NOTHING;
        """
        db.execute_query(query, (tid, cid, amt))

    loans = [
        (101, 1, 5000.00, 4.5, "active"),
        (102, 1, 12000.00, 3.8, "paid_off"),
        (103, 2, 8500.00, 5.2, "active"),
        (104, 2, 2000.00, 6.0, "paid_off"),
        (105, 3, 15000.00, 7.5, "defaulted"),
        (106, 3, 3000.00, 5.0, "active"),
    ]
    for lid, cid, princ, rate, status in loans:
        query = """
            INSERT INTO loans (loan_id, customer_id, principal, interest_rate, status)
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (loan_id) DO NOTHING;
        """
        db.execute_query(query, (lid, cid, princ, rate, status))

    logger.info("Seed data populated successfully.")



In [50]:
#task 3 and for - extraction, pure transformation, loading and verification

# pure Transaction Functions (Decoupled from Database) - imported from transform.py
# so the same logic that is unit-tested in test_transform.py is what actually runs here
from transform import compute_loan_metrics, categorize_customer, check_high_value_flag


# ETL Pipeline Methods

def extract_data(db: DBConnection):
    """
    Extracts raw customer, transaction, and loan records from Postgresql 
    """
    cust_rows = db.execute_query(
        "SELECT customer_id, name FROM customers;", fetch=True
    )
    tx_rows = db.execute_query(
        "SELECT customer_id, amount FROM transactions;", fetch=True
    )
    loan_rows = db.execute_query(
        "SELECT customer_id, principal, status FROM loans;", fetch=True
    )

    customers = [{"customer_id": r[0], "name": r[1]} for r in cust_rows]
    transactions = [{"customer_id": r[0], "amount": r[1]} for r in tx_rows]
    loans = [
        {"customer_id": r[0], "principal": r[1], "status": r[2]}
        for r in loan_rows
    ]
    return customers, transactions, loans

def transform_data(customers, transactions, loans):
    """
    Transforms raw records into account summary representations using pure logic. 
    """
    # summarize transactions per customer
    tx_totals = {}
    for tx in transactions:
        cid = tx["customer_id"]
        tx_totals[cid] = tx_totals.get(cid, 0.0) + float(tx['amount'])

    loan_metrics = compute_loan_metrics(loans)

    summary_records = []
    for c in customers:
        cid = c['customer_id']
        total_tx = tx_totals.get(cid, 0.0)

        c_loan = loan_metrics.get(
            cid,
            {
                "total_loan_exposure": 0.0,
                "active_loan_count": 0,
                "has_defaulted": False
            }
        )

        category = categorize_customer(
            total_tx,
            has_defaulted_loan=c_loan['has_defaulted']
        )

        summary_records.append(
            {
                "customer_id": cid,
                "total_transactions": total_tx,
                "total_loan_exposure": c_loan["total_loan_exposure"],
                "active_loan_count": c_loan["active_loan_count"],
                "category": category,
            }
        )
    return summary_records


In [51]:
# task 4

def load_summary(db: DBConnection, summary_records):
    """ 
    Upserts summary records into the account_summary table using parameterized SQL.
    """
    upsert_query = """ 
        INSERT INTO account_summary (
            customer_id, total_transactions, total_loan_exposure, active_loan_count, category, updated_at
        )
        VALUES (%s, %s, %s, %s, %s, CURRENT_TIMESTAMP)
        ON CONFLICT (customer_id) DO UPDATE SET
            total_transactions = EXCLUDED.total_transactions,
            total_loan_exposure = EXCLUDED.total_loan_exposure,
            active_loan_count = EXCLUDED.active_loan_count,
            category = EXCLUDED.category,
            updated_at = CURRENT_TIMESTAMP;
    """

    for rec in summary_records:
        db.execute_query(
            upsert_query,
            (
                rec["customer_id"],
                rec["total_transactions"],
                rec["total_loan_exposure"],
                rec["active_loan_count"],
                rec["category"],
            )
        )
    logger.info(f"Loaded/Upserted {len(summary_records)} summary records.")


def verify(db: DBConnection):
    """Prints a reconciliation report and verifies customer reconciliation parity."""
    # 1. Fetch reconciliation data joining customers and account_summary
    query = """
        SELECT 
            c.customer_id,
            c.name,
            COALESCE(a.total_transactions, 0.00),
            COALESCE(a.total_loan_exposure, 0.00),
            COALESCE(a.active_loan_count, 0),
            COALESCE(a.category, 'UNMAPPED')
        FROM customers c
        LEFT JOIN account_summary a ON c.customer_id = a.customer_id
        ORDER BY c.customer_id;
    """
    results = db.execute_query(query, fetch=True)

    # 2. Print simple summary table
    print("\n--- RECONCILIATION REPORT ---")
    print(
        "ID | Name               | Total Tx ($) | Exposure ($) | Active Loans | Category"
    )
    print("-" * 75)

    for cid, name, tx, exp, loans_cnt, cat in results:
        print(
            f"{cid:<2} | {name:<18} | ${tx:<11.2f} | ${exp:<11.2f} | {loans_cnt:<12} | {cat}"
        )

    # 3. Check customer count vs account summary count parity
    total_cust = db.execute_query(
        "SELECT COUNT(*) FROM customers;", fetch=True
    )[0][0]
    total_summ = db.execute_query(
        "SELECT COUNT(*) FROM account_summary;", fetch=True
    )[0][0]

    assert (
        total_cust == total_summ
    ), f"Mismatch: {total_cust} customers vs {total_summ} summary rows!"

    print(f"\nVERIFICATION SUCCESSFUL: All {total_cust} customers reconciled.")

In [52]:
# pipeline execution and connection failure simulation

# instantiate db connection
db = DBConnection(DB_CONFIG)
try:
    print("Step 1: Setting up schema and seed data ...")
    create_scema(db)
    seed_data(db)

    print("\n step 2: executing extract - tranform - load pipeline...")

    customers, transactions, loans = extract_data(db)
    transformed_records = transform_data(customers, transactions, loans)
    load_summary(db, transformed_records)

    print("\n step 3: running verification...")
    verify(db)

    print("\n step 4: simulating a real transient connection failure (Task 2 demo)")
    # Point a second DBConnection at an unreachable port so psycopg2.connect()
    # genuinely raises OperationalError inside execute_query(). This exercises
    # the retry_db_operation decorator for real: it will log 3 retry attempts,
    # attempt a rollback each time, and finally re-raise after max_retries.
    bad_config = {**DB_CONFIG, "port": 5999}
    bad_db = DBConnection(bad_config)
    try:
        bad_db.execute_query("SELECT 1;", fetch=True)
    except OperationalError as e:
        logger.error(f"Simulated failure confirmed after retries exhausted: {e}")
        print("Simulated failure handled: retries + rollback logged to etl.log")
    finally:
        bad_db.close()

    print("\n step 5: confirming the real connection still works after the failure")
    count = db.execute_query("SELECT COUNT(*) FROM customers;", fetch=True)
    print(f"customers table row count: {count[0][0]}")

finally:
    db.close()


2026-09-23 21:38:15,768 [INFO] Idempodent schema created / verified successfully.


2026-09-23 21:38:15,783 [INFO] Seed data populated successfully.
2026-09-23 21:38:15,794 [INFO] Loaded/Upserted 3 summary records.


Step 1: Setting up schema and seed data ...

 step 2: executing extract - tranform - load pipeline...

 step 3: running verification...

--- RECONCILIATION REPORT ---
ID | Name               | Total Tx ($) | Exposure ($) | Active Loans | Category
---------------------------------------------------------------------------
1  | Alice Smith        | $21000.00    | $5000.00     | 1            | Premium
2  | Bob Jones          | $7000.00     | $8500.00     | 1            | Standard
3  | Charlie Brown      | $1000.00     | $3000.00     | 1            | At Risk


2026-09-23 21:38:15,809 [WARNING] OperationalError encountered on attempt 1/3: connection to server at "localhost" (127.0.0.1), port 5999 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?




VERIFICATION SUCCESSFUL: All 3 customers reconciled.

 step 4: simulating a real transient connection failure (Task 2 demo)


2026-09-23 21:38:16,812 [WARNING] OperationalError encountered on attempt 2/3: connection to server at "localhost" (127.0.0.1), port 5999 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

2026-09-23 21:38:17,818 [WARNING] OperationalError encountered on attempt 3/3: connection to server at "localhost" (127.0.0.1), port 5999 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

2026-09-23 21:38:17,822 [ERROR] Max retries reached.
2026-09-23 21:38:17,827 [ERROR] Simulated failure confirmed after retries exhausted: connection to server at "localhost" (127.0.0.1), port 5999 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?



Simulated failure handled: retries + rollback logged to etl.log

 step 5: confirming the real connection still works after the failure
customers table row count: 3
